# The Final RAG Pipeline: Grounding AI in Private Data
We are moving from "Chatting with an AI" to "Building a Knowledge Engine." 

**The RAG Workflow:**
1. **Ingest:** Load and chunk our private data (*Hamlet*).
2. **Index:** Convert those chunks into embeddings (vectors).
3. **Retrieve:** When a user asks a question, find the most relevant chunks.
4. **Augment:** Stuff those chunks into the LLM's prompt as "Context."
5. **Generate:** The LLM answers the question based *only* on the provided context.

In [7]:
import os
import numpy as np
import gradio as gr
from openai import OpenAI
from litellm import completion
from dotenv import load_dotenv

load_dotenv(override=True)
client = OpenAI()

print("🏗️ RAG Pipeline Environment Initialized!")

🏗️ RAG Pipeline Environment Initialized!


## Part 1: Preparing the Knowledge Base

We load the text, break it into 1,000-character chunks, and convert them into vectors. In a real system, this "Indexing" phase would happen once and be stored in a database.

In [12]:
# 1. Load Document
with open("hamlet.txt", "r", encoding="utf-8") as f:
    text = f.read()

# 2. Simple Recursive Chunking
chunk_size = 1000
overlap = 150
chunks = []
for i in range(0, len(text), chunk_size - overlap):
    chunks.append(text[i:i + chunk_size])

print(f"📊 Created {len(chunks)} chunks.")

# 3. Vectorize (Indexing)
def get_embedding(text):
    return client.embeddings.create(input=[text], model="text-embedding-3-small").data[0].embedding

print("🧠 Embedding knowledge base... (this takes a few seconds)")
chunk_vectors = [get_embedding(c) for c in chunks]
print("✅ Indexing Complete.")

📊 Created 226 chunks.
🧠 Embedding knowledge base... (this takes a few seconds)
✅ Indexing Complete.


## Part 2: The Retrieval Engine

We use Cosine Similarity to find the 3 most relevant "Knowledge Nuggets" for any user query.

In [13]:
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def get_relevant_context(query, top_k=5):
    query_vec = get_embedding(query)
    scores = [cosine_similarity(query_vec, cv) for cv in chunk_vectors]
    
    # Get indices of top_k highest scores
    top_indices = np.argsort(scores)[-top_k:][::-1]
    
    # DIAGNOSTIC: Print top score
    print(f"🔍 Retrieval Match: Top Score = {scores[top_indices[0]]:.4f}")
    
    return [chunks[i] for i in top_indices]

## Part 3: The RAG Brain (Streaming & Grounding)

We construct a prompt that forces the AI to use the retrieved context. We then stream the result to the UI.

In [14]:
def rag_chat(message, history):
    # 1. Retrieve the Facts
    relevant_chunks = get_relevant_context(message)
    context_block = "\n---\n".join(relevant_chunks)
    
    # 2. Build the Grounded Prompt
    system_prompt = f"""
    You are a Shakespearean Scholar. 
    Answer the user's question using the provided context from the play Hamlet.
    Be helpful and descriptive, but stay strictly grounded in the provided text.
    If the answer is absolutely not in the context, say 'Alas, the text provides no answer to this query.'
    
    CONTEXT:
    {context_block}
    """
    
    messages = [{"role": "system", "content": system_prompt}]
    
    # 3. Add History (Robust handling for all Gradio versions)
    for msg in history:
        if hasattr(msg, 'role'):
            messages.append({"role": msg.role, "content": msg.content})
        elif isinstance(msg, dict):
            messages.append({"role": msg['role'], "content": msg['content']})
        elif isinstance(msg, (list, tuple)) and len(msg) == 2:
            messages.append({"role": "user", "content": msg[0]})
            messages.append({"role": "assistant", "content": msg[1]})
    
    messages.append({"role": "user", "content": message})
    
    # 4. Stream the grounded response
    response = completion(model="openai/gpt-4o-mini", messages=messages, stream=True)
    
    partial_text = ""
    for chunk in response:
        token = chunk.choices[0].delta.content
        if token:
            partial_text += token
            yield partial_text

## Part 4: The Capstone UI

We wrap our RAG logic in a `gr.ChatInterface` to create a professional, grounded application.

In [ ]:
demo = gr.ChatInterface(
    fn=rag_chat, 
    title="📖 Hamlet: The Grounded Scholar",
    description="Ask me anything about the play. I will only answer based on the actual text!",
    examples=["Who killed Hamlet's father?", "What is the meaning of 'To be or not to be'?"]
)

print("🚀 RAG Pipeline Wired and Ready.")
demo.launch()

🚀 RAG Pipeline Wired and Ready.
* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


🔍 Retrieval Match: Top Score = 0.4289
🔍 Retrieval Match: Top Score = 0.6220
🔍 Retrieval Match: Top Score = 0.5672
🔍 Retrieval Match: Top Score = 0.4122
